# Logistic Regression News Topic Classification
## Arseniy Uspenskiy
## USPARS001

Importing necessary modules

In [ ]:
# general imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import random 
import os
from typing import Optional

# pytorch
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

# scikit learn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay

# constants
SPLITS = ("train", "dev", "test")
METHODS = ("bow", "tfidf")

method for cleaning text data

In [3]:
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"htpp\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

Class for loading text of a language

In [4]:
class NewsDataLoader:

    def __init__(self, root, language):
        self.root = root
        self.language = language
        self.directory = os.path.join(root, language)
        self.labels = self._load_labels()
        self.label2id = {lab: i for i, lab in enumerate(self.labels)}

    def _load_labels(self):
        path = os.path.join(self.directory, "labels.txt")
        with open(path, encoding="utf-8") as f:
            return [line.strip() for line in f if line.strip()]

    def _load_split(self, split):
        path = os.path.join(self.directory, f"{split}.tsv")
        df = pd.read_csv(path, sep="\t", engine="python")

        df["full_text"] = (df["headline"].fillna("") + " " + df["text"].fillna("")).apply(clean_text)
        df["label_id"] = df["category"].map(self.label2id)
        df["label_id"] = df["label_id"].astype(int)

        return df

    def load_splits(self):
        return {split: self._load_split(split) for split in SPLITS}

Class for extracting features for perceptron training

In [5]:
class FeatureExtractor:

    def __init__(self, method, max_features: Optional[int] = None, min_df: int = 1):
        self.method = method
        if method == "bow":
            self.vectorizer = CountVectorizer(max_features=max_features, min_df=min_df)
        else:
            self.vectorizer = TfidfVectorizer(max_features=max_features, min_df=min_df)
        
    def transform_train(self, train):
        return self.vectorizer.fit_transform(train)

    def transform(self, text):
        return self.vectorizer.transform(text)

    def vocab_size(self):
        return len(self.vectorizer.vocabulary_)

    def feature_names(self):
        return self.vectorizer.get_feature_names_out()

Method for using the two loader classes for building the full dataset of a language

In [6]:
def build_dataset(root, language, method, max_features: Optional[int] = None, min_df: int = 1):

    loader = NewsDataLoader(root, language)
    splits = loader.load_splits()
    extractor = FeatureExtractor(method, max_features=max_features, min_df=min_df)

    X_train = extractor.transform_train(splits["train"]["full_text"])
    X_val = extractor.transform(splits["dev"]["full_text"])
    X_test = extractor.transform(splits["test"]["full_text"])

    return X_train, X_val, X_test

def y_values(root, language):

    loader = NewsDataLoader(root, language)
    splits = loader.load_splits()

    return splits["train"]["label_id"].to_numpy(), splits["dev"]["label_id"].to_numpy(), splits["test"]["label_id"].to_numpy()

Multinomial logistic regression implementation

In [7]:
class MultinomialLogisticRegression(nn.Module):

    def __init__(self, input_size, num_classes):
        super().__init__()
        self.linear = nn.Linear(input_size, num_classes)

    def forward(self, features):
        return self.linear(features)

    def compute_probabilities(self, features):
        logits = self.forward(features)
        return F.softmax(logits, dim=1)

    def predict(self, features):
        probabilities = self.compute_probabilities(features)
        return torch.argmax(probabilities, dim=1)

Load all language datasets

In [8]:
eng_X_train_bow, eng_X_val_bow, eng_X_train_bow = build_dataset("news-dataset", "eng", "bow")
eng_X_train_tfidf, eng_X_val_tfidf, eng_X_train_tfidf = build_dataset("news-dataset", "eng", "tfidf")
eng_y_train, eng_y_val, eng_y_train = y_values("news-dataset", "eng")

sna_X_train_bow, sna_X_val_bow, sna_X_train_bow = build_dataset("news-dataset", "sna", "bow")
sna_X_train_tfidf, sna_X_val_tfidf, sna_X_train_tfidf = build_dataset("news-dataset", "sna", "tfidf")
sna_y_train, sna_y_val, sna_y_train = y_values("news-dataset", "sna")

xho_X_train_bow, xho_X_val_bow, xho_X_train_bow = build_dataset("news-dataset", "xho", "bow")
xho_X_train_tfidf, xho_X_val_tfidf, xho_X_train_tfidf = build_dataset("news-dataset", "xho", "tfidf")
xho_y_train, xho_y_val, xho_y_train = y_values("news-dataset", "xho")

Methods for training and evaluation of models

In [10]:
class TrainHistory:

    def __init__(self):
        self.train_loss = []
        self.val_loss = []
        self.val_accuracy = []

def evaluate(model: MultinomialLogisticRegression, loader, device):

    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device).long()
            logits = model(features)

            loss = F.cross_entropy(logits, labels, reduction="sum")
            total_loss += loss.item()

            preds = model.predict(features)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

def train_model(model: MultinomialLogisticRegression, train_data, val_data, device, learning_rate: float = 0.01, batch_size: int = 32, num_epochs: int = 50, patience: int = 5):

    model = model.to(device)

    train = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val = DataLoader(val_data, batch_size=batch_size, shuffle=False)

    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

    history = TrainHistory()
    best_acc = -1.0
    no_improvement_epochs = 0

    for epoch in range(num_epochs):
        model.train()
        cur_loss = 0.0
        n = 0

        for features, labels in train:
            features, labels = features.to(device), labels.to(device)

            optimizer.zero_grad()
            logits = model(features)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            n += labels.size(0)

        train_loss = cur_loss / n
        val_loss, val_acc = evaluate(model, val, device)

        history.train_loss.append(train_loss)
        history.val_loss.append(val_loss)
        history.val_accuracy.append(val_acc)

        print(f"Epoch [{epoch + 1}/{num_epochs}]\ntrain_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            no_improvement_epochs = 0
        else:
            no_improvement_epochs += 1
            if no_improvement_epochs >= patience:
                print("Early stopping due to exceeding patience")
                break

    return history

Create hyperparameter testing arrays

In [ ]:
# creating tests

Training Perceptrons

In [ ]:
# Training code